<a href="https://colab.research.google.com/github/Dev-anono/notebooks/blob/main/TripoSR_gradio_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /content
!rm -rf TripoSR TorchMCubes

# Клонируем TripoSR
!git clone https://github.com/VAST-AI-Research/TripoSR.git

# Установка базовых зависимостей
!pip install -U "numpy>=2.0.0" "scipy>=1.13.0" trimesh omegaconf einops rembg gradio onnxruntime-gpu
!pip install -e /content/TripoSR

# Установка torch-mcubes из исходников для поддержки Python 3.12
!git clone https://github.com/tatsy/torchmcubes.git
%cd /content/torchmcubes
!pip install .
%cd /content

# Веса модели
!mkdir -p /content/model
!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/stabilityai/TripoSR/resolve/main/model.ckpt -d /content/model -o model.ckpt

# Финальная перезагрузка для применения изменений
import os
os.kill(os.getpid(), 9)

/content
Cloning into 'TripoSR'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 161 (delta 63), reused 41 (delta 41), pack-reused 66 (from 1)
Receiving objects: 100% (161/161), 36.71 MiB | 32.46 MiB/s, done.
Resolving deltas: 100% (65/65), done.
Obtaining file:///content/TripoSR
ERROR: file:///content/TripoSR does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Cloning into 'torchmcubes'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 199 (delta 19), reused 16 (delta 16), pack-reused 169 (from 1)
Receiving objects: 100% (199/199), 462.47 KiB | 3.79 MiB/s, done.
Resolving deltas: 100% (114/114), done.
/content/torchmcubes
Processing /content/torchmcubes
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Prepari

In [15]:
import sys
import os
import torch
import numpy as np
from PIL import Image
from rembg import remove, new_session

# Добавляем путь к TripoSR
tripo_path = '/content/TripoSR'
sys.path.append(tripo_path)

from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

# Настройки
input_path = '/content/IMG_20260421_110518_378.jpg'
output_dir = '/content/output'
os.makedirs(output_dir, exist_ok=True)

# 1. Удаление фона и подготовка изображения
print("Обработка изображения...")
rembg_session = new_session()
image = Image.open(input_path)
image = remove_background(image, rembg_session)
image = resize_foreground(image, 0.85)

img_array = np.array(image).astype(np.float32) / 255.0
if img_array.shape[-1] == 4:
    alpha = img_array[:, :, 3:4]
    img_array = img_array[:, :, :3] * alpha + (1 - alpha) * 0.5

image = Image.fromarray((img_array * 255.0).astype(np.uint8))
image.save(os.path.join(output_dir, "input.png"))

# 2. Загрузка модели
print("Загрузка модели TripoSR...")
# Используем официальный метод загрузки.
# Он сам найдет нужные конфиги в Hugging Face или кэше.
model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt"
)

# Принудительно загружаем именно наши скачанные веса, если нужно
ckpt_path = "/content/model/model.ckpt"
state_dict = torch.load(ckpt_path, map_location="cpu")
if "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]
model.load_state_dict(state_dict)

model.to("cuda")
model.eval()

# 3. Генерация 3D
print("Генерация 3D модели...")
with torch.no_grad():
    scene_codes = model([image], device="cuda")
    meshes = model.extract_mesh(scene_codes, resolution=256)

# 4. Сохранение результата
output_obj = os.path.join(output_dir, "mesh.obj")
meshes[0].export(output_obj)

print(f"Готово! 3D модель сохранена по пути: {output_obj}")

Обработка изображения...


Загрузка модели TripoSR...


config.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Генерация 3D модели...


TypeError: TSR.extract_mesh() missing 1 required positional argument: 'has_vertex_color'

In [18]:
import traceback
import gc

try:
    # Очистка памяти перед запуском
    torch.cuda.empty_cache()
    gc.collect()

    print("Запуск экстракции меша (разрешение 128 для экономии памяти)... ")

    # Настройка размера чанка для экономии VRAM
    model.renderer.chunk_size = 4096

    with torch.no_grad():
        scene_codes = model([image], device='cuda')
        # Снижаем resolution до 128 для успешного прохождения по памяти
        meshes = model.extract_mesh(scene_codes, resolution=128, has_vertex_color=True)

    output_obj = os.path.join(output_dir, 'mesh.obj')
    meshes[0].export(output_obj)
    print(f'\nУспешно! 3D модель сохранена: {output_obj}')
    print("Вы можете скачать её из папки output в левой панели (иконка папки).")

except torch.OutOfMemoryError:
    print('\nОШИБКА: Все еще не хватает памяти. Попробуйте нажать Runtime -> Restart session и запустить только блоки генерации.')
except Exception as e:
    print('\nПроизошла ошибка:')
    traceback.print_exc()

Запуск экстракции меша (разрешение 128 для экономии памяти)... 

Успешно! 3D модель сохранена: /content/output/mesh.obj
Вы можете скачать её из папки output в левой панели (иконка папки).
